# MPEDANet 結果視覺化工具

本程式可以將訓練出來的 MPEDANet 預測結果與 Ground Truth 一起呈現在原始影像上，並透過 Scrollbar 切換不同的橫斷面 (transverse slices)。

In [22]:
import os
import numpy as np
import torch
import SimpleITK as sitk
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
from torchvision.transforms import transforms
from ipywidgets import interact, IntSlider, Checkbox, Layout
import ipywidgets as widgets
from IPython.display import display

# 導入模型
from networks.MyNet.MyNet import MyNet

# 定義數據轉換類別
class CenterCrop(object):
    """中心裁剪類別"""
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        (c, w, h, d) = image.shape

        w1 = int(round((w - self.output_size[0]) / 2.))
        h1 = int(round((h - self.output_size[1]) / 2.))
        d1 = int(round((d - self.output_size[2]) / 2.))

        label = label[w1:w1 + self.output_size[0], h1:h1 + self.output_size[1], d1:d1+self.output_size[2]]
        image = image[:, w1:w1 + self.output_size[0], h1:h1 + self.output_size[1], d1:d1+self.output_size[2]]
        
        return {'image': image, 'label': label}

class ToTensor(object):
    """轉換為 Tensor 類別"""
    def __call__(self, sample):
        image = sample['image']
        label = sample['label']
        
        image = torch.from_numpy(image).float()
        label = torch.from_numpy(label).long()
        
        return {'image': image, 'label': label}

In [23]:
# 設置 matplotlib 顯示模式
%matplotlib inline
import warnings

warnings.filterwarnings('ignore')

print("環境設置完成！")

環境設置完成！


## 1. 載入資料與模型的輔助函數

In [24]:
modalities = ('_t1.nii.gz', '_t1ce.nii.gz', '_t2.nii.gz', '_flair.nii.gz')

def load_data(data_path, patch_size=(160, 160, 64)):
    """載入原始影像、標籤（Ground Truth）"""
    
    # 載入標籤
    label = sitk.GetArrayFromImage(
        sitk.ReadImage(data_path + '/' + f'{data_path[-15:]}_seg.nii.gz')).transpose(1, 2, 0)
    
    # 載入並堆疊四種模態的影像
    images = np.stack([
        sitk.GetArrayFromImage(
            sitk.ReadImage(data_path + '/' + f'{data_path[-15:]}{modal}')).transpose(1, 2, 0) 
        for modal in modalities
    ], 0)
    
    # 數據類型轉換
    label = label.astype(np.uint8)
    images = images.astype(np.float32)
    
    # 標準化處理（與訓練時一致）
    mask = images.sum(0) > 0
    for k in range(4):
        x = images[k, ...]
        y = x[mask]
        x[mask] -= y.mean()
        x[mask] /= y.std()
        images[k, ...] = x
    
    # 標籤轉換 [0,1,2,4] -> [0,1,2,3]
    label[label == 4] = 3
    
    # 應用中心裁剪和轉張量
    sample = {'image': images, 'label': label}
    trans = transforms.Compose([CenterCrop(patch_size), ToTensor()])
    sample = trans(sample)
    
    return sample['image'], sample['label']

def load_raw_images(data_path, patch_size=(160, 160, 64)):
    """載入未標準化的原始影像（用於顯示）"""
    
    # 載入四種模態的原始影像
    images = np.stack([
        sitk.GetArrayFromImage(
            sitk.ReadImage(data_path + '/' + f'{data_path[-15:]}{modal}')).transpose(1, 2, 0) 
        for modal in modalities
    ], 0)
    
    images = images.astype(np.float32)
    label = sitk.GetArrayFromImage(
        sitk.ReadImage(data_path + '/' + f'{data_path[-15:]}_seg.nii.gz')).transpose(1, 2, 0)
    label = label.astype(np.uint8)
    label[label == 4] = 3
    
    sample = {'image': images, 'label': label}
    trans = transforms.Compose([CenterCrop(patch_size), ToTensor()])
    sample = trans(sample)
    
    return sample['image']

def load_model(model_path, device):
    """載入訓練好的模型"""
    model = MyNet(in_channels=4, num_classes=4).to(device)
    
    if os.path.exists(model_path):
        weight_dict = torch.load(model_path, map_location=device)
        model.load_state_dict(weight_dict['model'])
        print(f'成功載入模型權重: {model_path}')
    else:
        print(f'警告: 找不到模型權重檔案 {model_path}')
        return None
    
    model.eval()
    return model

def get_prediction(model, image, device):
    """使用模型進行預測"""
    with torch.no_grad():
        image_tensor = image.unsqueeze(0).to(device)
        output = model(image_tensor)
        prediction = torch.argmax(output, dim=1)
        prediction = prediction.squeeze(0).cpu().numpy()
    return prediction

## 2. 視覺化函數

In [25]:
def create_overlay(background, mask, color, alpha=0.5):
    """創建覆蓋層，將分割遮罩套用在背景影像上"""
    overlay = background.copy()
    if len(overlay.shape) == 2:
        overlay = np.stack([overlay, overlay, overlay], axis=-1)
    
    # 將遮罩區域著色
    for i in range(3):
        overlay[:, :, i] = np.where(mask > 0, 
                                     overlay[:, :, i] * (1 - alpha) + color[i] * alpha * 255,
                                     overlay[:, :, i])
    return overlay

def visualize_slice(raw_image, gt_label, pred_label, slice_idx, 
                    show_gt=True, show_pred=True, modality_idx=0):
    """視覺化單一橫斷面切片"""
    
    # 轉換為 numpy array（如果是 Tensor）
    if torch.is_tensor(raw_image):
        raw_image = raw_image.cpu().numpy()
    if torch.is_tensor(gt_label):
        gt_label = gt_label.cpu().numpy()
    if torch.is_tensor(pred_label):
        pred_label = pred_label.cpu().numpy()
    
    # 提取對應的切片
    img_slice = raw_image[modality_idx, :, :, slice_idx].copy()
    gt_slice = gt_label[:, :, slice_idx].copy()
    pred_slice = pred_label[:, :, slice_idx].copy()
    
    # 正規化影像以便顯示
    if img_slice.max() > img_slice.min():
        img_slice = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min())
        img_slice = (img_slice * 255).astype(np.uint8)
    else:
        img_slice = np.zeros_like(img_slice, dtype=np.uint8)
    
    # 定義不同腫瘤區域的顏色 (RGB格式)
    colors = {
        1: [255, 0, 0],    # 壞疽 - 紅色
        2: [0, 255, 0],    # 浮腫 - 綠色  
        3: [0, 0, 255]     # 增強腫瘤 - 藍色
    }
    
    # 創建圖形 - 增大字體
    plt.close('all')
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    
    modality_names = ['T1', 'T1CE', 'T2', 'FLAIR']
    
    # 1. 原始影像
    axes[0].imshow(img_slice, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title(f'{modality_names[modality_idx]} - Slice {slice_idx}', fontsize=20, fontweight='bold')
    axes[0].axis('off')
    
    # 2. Ground Truth 覆蓋
    if show_gt:
        # 創建彩色覆蓋圖
        gt_color = np.stack([img_slice, img_slice, img_slice], axis=-1).copy()
        for label_val, color in colors.items():
            mask = (gt_slice == label_val)
            if np.any(mask):
                for i in range(3):
                    gt_color[:, :, i][mask] = (gt_color[:, :, i][mask] * 0.5 + color[i] * 0.5).astype(np.uint8)
        axes[1].imshow(gt_color)
    else:
        axes[1].imshow(img_slice, cmap='gray', vmin=0, vmax=255)
    
    axes[1].set_title('Ground Truth (GT)', fontsize=20, fontweight='bold')
    axes[1].axis('off')
    
    # 3. 預測結果覆蓋
    if show_pred:
        # 創建彩色覆蓋圖
        pred_color = np.stack([img_slice, img_slice, img_slice], axis=-1).copy()
        for label_val, color in colors.items():
            mask = (pred_slice == label_val)
            if np.any(mask):
                for i in range(3):
                    pred_color[:, :, i][mask] = (pred_color[:, :, i][mask] * 0.5 + color[i] * 0.5).astype(np.uint8)
        axes[2].imshow(pred_color)
    else:
        axes[2].imshow(img_slice, cmap='gray', vmin=0, vmax=255)
    
    axes[2].set_title('Prediction', fontsize=20, fontweight='bold')
    axes[2].axis('off')
    
    # 添加圖例 - 增大字體
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='red', alpha=0.7, label='Necrotic Tumor'),
        Patch(facecolor='green', alpha=0.7, label='Edema'),
        Patch(facecolor='blue', alpha=0.7, label='Enhancing Tumor')
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=16, frameon=False)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.1)
    
    # 輸出統計資訊
    gt_counts = {1: np.sum(gt_slice == 1), 2: np.sum(gt_slice == 2), 3: np.sum(gt_slice == 3)}
    pred_counts = {1: np.sum(pred_slice == 1), 2: np.sum(pred_slice == 2), 3: np.sum(pred_slice == 3)}
    
    print(f"\n=== 切片 {slice_idx} 統計資訊 ===")
    print(f"GT   - 壞疽: {gt_counts[1]:5d} px, 浮腫: {gt_counts[2]:5d} px, 增強腫瘤: {gt_counts[3]:5d} px")
    print(f"Pred - 壞疽: {pred_counts[1]:5d} px, 浮腫: {pred_counts[2]:5d} px, 增強腫瘤: {pred_counts[3]:5d} px")
    
    # 檢查是否有病灶
    if sum(gt_counts.values()) == 0:
        print("⚠️  此切片沒有 Ground Truth 標註的病灶")
    if sum(pred_counts.values()) == 0:
        print("⚠️  此切片沒有預測到任何病灶")
    
    plt.show()
    
    return fig

## 3. 互動式視覺化（使用 ipywidgets）

In [26]:
class InteractiveViewer:
    """互動式視覺化工具 - 支援多病例選擇"""
    
    def __init__(self, model, device, data_base_path, patient_list, patch_size=(160, 160, 64)):
        """初始化視覺化工具
        
        Args:
            model: 訓練好的模型
            device: 計算設備 (cpu/cuda)
            data_base_path: 數據基礎路徑
            patient_list: 病例列表 (例如: ['00072', '00073', '00074'])
            patch_size: 影像裁剪大小
        """
        self.model = model
        self.device = device
        self.data_base_path = data_base_path
        self.patient_list = patient_list
        self.patch_size = patch_size
        
        # 當前數據
        self.current_patient = None
        self.raw_image = None
        self.gt_label = None
        self.pred_label = None
        self.num_slices = 0
        
        # 創建病例選擇下拉框
        self.patient_dropdown = widgets.Dropdown(
            options=[(f'病例 {p}', p) for p in patient_list],
            value=patient_list[0],
            description='選擇病例:',
            style={'description_width': '80px'},
            layout=Layout(width='300px')
        )
        
        # 創建其他控件
        self.slice_slider = IntSlider(
            value=0,
            min=0,
            max=63,
            step=1,
            description='切片:',
            continuous_update=False,
            style={'description_width': '80px'},
            layout=Layout(width='80%')
        )
        
        self.modality_dropdown = widgets.Dropdown(
            options=[('T1', 0), ('T1CE', 1), ('T2', 2), ('FLAIR', 3)],
            value=1,
            description='影像模態:',
            style={'description_width': '80px'},
            layout=Layout(width='300px')
        )
        
        self.show_gt_checkbox = Checkbox(
            value=True,
            description='顯示 Ground Truth',
            indent=False
        )
        
        self.show_pred_checkbox = Checkbox(
            value=True,
            description='顯示預測結果',
            indent=False
        )
        
        # 狀態顯示
        self.status_label = widgets.HTML(value="<b>狀態:</b> 初始化中...")
        
        # 圖片顯示區域
        self.output = widgets.Output()
        
    def load_patient_data(self, patient_id):
        """載入指定病例的數據"""
        with self.output:
            print(f"\n正在載入病例 {patient_id} 的數據...")
        
        try:
            # 構建數據路徑
            data_path = f"{self.data_base_path}/BraTS2021_{patient_id}"
            
            # 載入數據
            image, gt_label = load_data(data_path, self.patch_size)
            raw_image = load_raw_images(data_path, self.patch_size)
            
            # 進行預測
            pred_label = get_prediction(self.model, image, self.device)
            
            # 更新內部狀態
            self.current_patient = patient_id
            self.raw_image = raw_image
            self.gt_label = gt_label
            self.pred_label = pred_label
            self.num_slices = raw_image.shape[3]
            
            # 更新滑桿範圍
            self.slice_slider.max = self.num_slices - 1
            self.slice_slider.value = self.num_slices // 2
            
            self.status_label.value = f"<b>狀態:</b> <span style='color:green;'>✓ 病例 {patient_id} 載入成功</span>"
            
            with self.output:
                print(f"✓ 病例 {patient_id} 載入完成！")
                print(f"  影像形狀: {raw_image.shape}")
                print(f"  切片數量: {self.num_slices}")
            
        except Exception as e:
            self.status_label.value = f"<b>狀態:</b> <span style='color:red;'>✗ 載入失敗: {str(e)}</span>"
            with self.output:
                print(f"✗ 載入失敗: {str(e)}")
    
    def on_patient_change(self, change):
        """當病例選擇改變時的回調"""
        new_patient = change['new']
        if new_patient != self.current_patient:
            with self.output:
                self.output.clear_output(wait=True)
            self.load_patient_data(new_patient)
            self.update_view()
    
    def update_view(self, change=None):
        """更新視圖"""
        if self.raw_image is None:
            return
            
        slice_idx = self.slice_slider.value
        modality_idx = self.modality_dropdown.value
        show_gt = self.show_gt_checkbox.value
        show_pred = self.show_pred_checkbox.value

        with self.output:
            self.output.clear_output(wait=True)
            visualize_slice(self.raw_image, self.gt_label, self.pred_label, 
                          slice_idx, show_gt, show_pred, modality_idx)
    
    def display(self):
        """顯示互動式介面"""
        # 綁定事件
        self.patient_dropdown.observe(self.on_patient_change, names='value')
        self.slice_slider.observe(self.update_view, names='value')
        self.modality_dropdown.observe(self.update_view, names='value')
        self.show_gt_checkbox.observe(self.update_view, names='value')
        self.show_pred_checkbox.observe(self.update_view, names='value')
        
        # 佈局設計
        controls_box = widgets.VBox([
            widgets.HTML(value="<h3>🏥 MPEDANet 結果視覺化</h3>"),
            self.status_label,
            widgets.HTML(value="<hr>"),
            self.patient_dropdown,
            self.modality_dropdown,
            self.slice_slider,
            widgets.HBox([self.show_gt_checkbox, self.show_pred_checkbox]),
        ], layout=Layout(padding='10px'))
        
        ui = widgets.VBox([
            controls_box,
            self.output
        ])
        
        # 顯示介面
        display(ui)
        
        # 載入第一個病例
        self.load_patient_data(self.patient_list[0])
        self.update_view()

## 4. 主程式 - 設定與執行

請根據您的實際情況修改以下參數：
- `case`: 病例編號
- `model_path`: 訓練好的模型權重路徑
- `data_path`: 病例資料所在路徑

In [27]:
# ==================== 設定參數 ====================
# 設定設備
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備: {device}")

# 病例編號和路徑
case = '00072'  # 修改為您要查看的病例編號
data_path = f"../dataset/brats2021/data/BraTS2021_{case}"
model_path = "/home/at0842/aaronwu901225master.ai13/sundries/MPEDANet-pytorch-main/TempCode/code_pytorch/checkpoint/MyNet/checkpoint90.pth"  # 修改為您的模型權重路徑

# 影像大小
patch_size = (160, 160, 64)

print(f"病例: {case}")
print(f"資料路徑: {data_path}")
print(f"模型路徑: {model_path}")

使用設備: cpu
病例: 00072
資料路徑: ../dataset/brats2021/data/BraTS2021_00072
模型路徑: /home/at0842/aaronwu901225master.ai13/sundries/MPEDANet-pytorch-main/TempCode/code_pytorch/checkpoint/MyNet/checkpoint90.pth


In [28]:
# ==================== 載入資料 ====================
print("\n正在載入資料...")

# 載入標準化後的影像（用於模型預測）
image, gt_label = load_data(data_path, patch_size)
print(f"標準化影像形狀: {image.shape}")
print(f"Ground Truth 標籤形狀: {gt_label.shape}")

# 載入原始影像（用於視覺化顯示）
raw_image = load_raw_images(data_path, patch_size)
print(f"原始影像形狀: {raw_image.shape}")

print(f"標籤中的唯一值: {np.unique(gt_label)}")
print("資料載入完成！")


正在載入資料...
標準化影像形狀: torch.Size([4, 160, 160, 64])
Ground Truth 標籤形狀: torch.Size([160, 160, 64])
原始影像形狀: torch.Size([4, 160, 160, 64])
標籤中的唯一值: [0 1 2 3]
資料載入完成！


In [29]:
# ==================== 載入模型並進行預測 ====================
print("\n正在載入模型...")
model = load_model(model_path, device)

if model is not None:
    print("\n正在進行預測...")
    pred_label = get_prediction(model, image, device)
    print(f"預測結果形狀: {pred_label.shape}")
    print(f"預測結果中的唯一值: {np.unique(pred_label)}")
    print("預測完成！")
else:
    print("模型載入失敗，無法進行預測")


正在載入模型...
成功載入模型權重: /home/at0842/aaronwu901225master.ai13/sundries/MPEDANet-pytorch-main/TempCode/code_pytorch/checkpoint/MyNet/checkpoint90.pth

正在進行預測...
預測結果形狀: (160, 160, 64)
預測結果中的唯一值: [0 1 2 3]
預測完成！


## 5a. (建議) 尋找有病灶的切片

這個步驟會幫助你找到包含腫瘤的切片位置

In [30]:
# 分析每個切片的病灶分布
if model is not None:
    print("正在分析切片病灶分布...\n")
    
    gt_np = gt_label.cpu().numpy() if torch.is_tensor(gt_label) else gt_label
    pred_np = pred_label if isinstance(pred_label, np.ndarray) else pred_label
    
    tumor_slices = []
    
    for slice_idx in range(gt_np.shape[2]):
        gt_slice = gt_np[:, :, slice_idx]
        pred_slice = pred_np[:, :, slice_idx]
        
        gt_tumor = np.sum(gt_slice > 0)
        pred_tumor = np.sum(pred_slice > 0)
        
        if gt_tumor > 0 or pred_tumor > 0:
            tumor_slices.append({
                'slice': slice_idx,
                'gt_pixels': gt_tumor,
                'pred_pixels': pred_tumor,
                'gt_class1': np.sum(gt_slice == 1),
                'gt_class2': np.sum(gt_slice == 2),
                'gt_class3': np.sum(gt_slice == 3),
            })
    
    if len(tumor_slices) > 0:
        print(f"找到 {len(tumor_slices)} 個包含腫瘤的切片\n")
        print("切片編號 | GT總像素 | Pred總像素 | GT壞疽 | GT浮腫 | GT增強")
        print("-" * 70)
        
        # 顯示前10個和後10個有病灶的切片
        display_slices = tumor_slices[:10] + (tumor_slices[-10:] if len(tumor_slices) > 10 else [])
        seen = set()
        
        for info in display_slices:
            if info['slice'] not in seen:
                print(f"  {info['slice']:3d}    | {info['gt_pixels']:7d}  | {info['pred_pixels']:9d}  | "
                      f"{info['gt_class1']:5d}  | {info['gt_class2']:5d}  | {info['gt_class3']:5d}")
                seen.add(info['slice'])
        
        # 找出腫瘤最多的切片
        max_tumor_slice = max(tumor_slices, key=lambda x: x['gt_pixels'])
        print(f"\n💡 建議查看切片 {max_tumor_slice['slice']} (GT 腫瘤像素最多: {max_tumor_slice['gt_pixels']} px)")
        
        # 找出包含所有三種類別的切片
        all_classes = [s for s in tumor_slices if s['gt_class1'] > 0 and s['gt_class2'] > 0 and s['gt_class3'] > 0]
        if all_classes:
            print(f"💡 切片 {all_classes[0]['slice']} 包含所有三種腫瘤類型")
    else:
        print("⚠️  警告：沒有找到任何包含腫瘤的切片！")
        print("這可能表示：")
        print("  1. 數據載入有問題")
        print("  2. 標籤全部為背景")
        print("  3. 模型預測全部為背景")

正在分析切片病灶分布...

找到 61 個包含腫瘤的切片

切片編號 | GT總像素 | Pred總像素 | GT壞疽 | GT浮腫 | GT增強
----------------------------------------------------------------------
    3    |      46  |         0  |     0  |    46  |     0
    4    |     153  |         0  |     0  |   153  |     0
    5    |     248  |         0  |     0  |   248  |     0
    6    |     365  |         0  |     0  |   365  |     0
    7    |     525  |         0  |     0  |   447  |    78
    8    |     675  |         0  |    20  |   456  |   199
    9    |     947  |       110  |    43  |   556  |   348
   10    |    1072  |       222  |    69  |   507  |   496
   11    |    1276  |       358  |   136  |   537  |   603
   12    |    1446  |       558  |   205  |   566  |   675
   54    |       0  |       856  |     0  |     0  |     0
   55    |       0  |       776  |     0  |     0  |     0
   56    |       0  |       610  |     0  |     0  |     0
   57    |       0  |       568  |     0  |     0  |     0
   58    |       0  |       

## 5. 啟動互動式視覺化介面

使用以下介面來探索不同的橫斷面切片：
- **切片滑桿**: 調整橫斷面的位置
- **影像模態**: 選擇要顯示的 MRI 模態（T1, T1CE, T2, FLAIR）
- **顯示選項**: 切換 Ground Truth 和預測結果的顯示

顏色說明：
- 🔴 **紅色**: 壞疽 (Necrotic Tumor Core)
- 🟢 **綠色**: 浮腫 (Edema)
- 🔵 **藍色**: 增強腫瘤 (Enhancing Tumor)

In [31]:
# ==================== 創建並顯示互動式視覺化介面 ====================
if model is not None:
    # 自動掃描數據目錄中的所有病例
    data_base_path = '../dataset/brats2021/data'
    
    print("正在掃描數據目錄...")
    patient_list = []
    
    if os.path.exists(data_base_path):
        # 列出所有以 BraTS2021_ 開頭的目錄
        all_items = os.listdir(data_base_path)
        for item in sorted(all_items):
            item_path = os.path.join(data_base_path, item)
            # 檢查是否為目錄且符合命名格式
            if os.path.isdir(item_path) and item.startswith('BraTS2021_'):
                # 提取病例編號（去掉 'BraTS2021_' 前綴）
                patient_id = item.replace('BraTS2021_', '')
                patient_list.append(patient_id)
        
        if len(patient_list) > 0:
            print(f"✓ 找到 {len(patient_list)} 個病例:")
            print(f"  {', '.join(patient_list[:10])}" + 
                  (f" ... (共 {len(patient_list)} 個)" if len(patient_list) > 10 else ""))
        else:
            print("⚠️ 警告: 沒有找到任何病例資料夾")
            print(f"請確認路徑 {data_base_path} 中包含 BraTS2021_* 格式的資料夾")
            patient_list = ['00072']  # 使用預設值
    else:
        print(f"⚠️ 警告: 路徑不存在 {data_base_path}")
        print("使用預設病例列表")
        patient_list = ['00072']
    
    # 創建支援多病例選擇的視覺化工具
    viewer = InteractiveViewer(
        model=model, 
        device=device, 
        data_base_path=data_base_path,
        patient_list=patient_list,
        patch_size=patch_size
    )
    viewer.display()
else:
    print("無法啟動視覺化介面，請確認模型載入成功")

正在掃描數據目錄...
✓ 找到 1251 個病例:
  00000, 00002, 00003, 00005, 00006, 00008, 00009, 00011, 00012, 00014 ... (共 1251 個)


## 6. (選用) 計算整體評估指標

計算 Dice 係數來評估預測結果的準確度。

In [32]:
def calculate_dice(pred, target, class_id):
    """計算特定類別的 Dice 係數"""
    # 確保轉換為 numpy array
    if torch.is_tensor(pred):
        pred = pred.cpu().numpy()
    if torch.is_tensor(target):
        target = target.cpu().numpy()
    
    pred_mask = (pred == class_id).astype(np.float32)
    target_mask = (target == class_id).astype(np.float32)
    
    intersection = np.sum(pred_mask * target_mask)
    union = np.sum(pred_mask) + np.sum(target_mask)
    
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    
    dice = (2.0 * intersection) / union
    return dice

if model is not None:
    print("\n=== 整體評估指標 ===")
    print(f"病例: {case}\n")
    
    # 轉換為 numpy array（如果是 Tensor）
    gt_label_np = gt_label.cpu().numpy() if torch.is_tensor(gt_label) else gt_label
    pred_label_np = pred_label if isinstance(pred_label, np.ndarray) else pred_label.cpu().numpy()
    
    # 計算各類別的 Dice 係數
    dice_scores = {}
    class_names = {
        1: "壞疽 (Necrotic Tumor)",
        2: "浮腫 (Edema)", 
        3: "增強腫瘤 (Enhancing Tumor)"
    }
    
    for class_id, class_name in class_names.items():
        dice = calculate_dice(pred_label_np, gt_label_np, class_id)
        dice_scores[class_name] = dice
        print(f"{class_name}: Dice = {dice:.4f}")
    
    # 計算複合指標
    # ET (Enhancing Tumor) = class 3
    # TC (Tumor Core) = class 1 + 3
    # WT (Whole Tumor) = class 1 + 2 + 3
    
    dice_et = calculate_dice(pred_label_np, gt_label_np, 3)
    
    pred_tc = ((pred_label_np == 1) | (pred_label_np == 3)).astype(np.float32)
    gt_tc = ((gt_label_np == 1) | (gt_label_np == 3)).astype(np.float32)
    intersection_tc = np.sum(pred_tc * gt_tc)
    union_tc = np.sum(pred_tc) + np.sum(gt_tc)
    dice_tc = (2.0 * intersection_tc) / union_tc if union_tc > 0 else 0.0
    
    pred_wt = (pred_label_np > 0).astype(np.float32)
    gt_wt = (gt_label_np > 0).astype(np.float32)
    intersection_wt = np.sum(pred_wt * gt_wt)
    union_wt = np.sum(pred_wt) + np.sum(gt_wt)
    dice_wt = (2.0 * intersection_wt) / union_wt if union_wt > 0 else 0.0
    
    print(f"\n=== BraTS 評估指標 ===")
    print(f"ET (Enhancing Tumor): Dice = {dice_et:.4f}")
    print(f"TC (Tumor Core): Dice = {dice_tc:.4f}")
    print(f"WT (Whole Tumor): Dice = {dice_wt:.4f}")
    print(f"\n平均 Dice: {(dice_et + dice_tc + dice_wt) / 3:.4f}")


=== 整體評估指標 ===
病例: 00072

壞疽 (Necrotic Tumor): Dice = 0.4724
浮腫 (Edema): Dice = 0.0251
增強腫瘤 (Enhancing Tumor): Dice = 0.2419

=== BraTS 評估指標 ===
ET (Enhancing Tumor): Dice = 0.2419
TC (Tumor Core): Dice = 0.3274
WT (Whole Tumor): Dice = 0.3567

平均 Dice: 0.3087
